# Домашнее задание 2 (10 баллов).

*Все задания ниже имеют равный вес*

Код для импорта мы написали за вас (не благодарите, нам не трудно). Дальше код будете писать вы.

[Тут](https://habr.com/ru/companies/ruvds/articles/494720/) шпора по pandas. За основу домашнего задания взят ноутбук [отсюда](https://rutube.ru/video/f884aa6ed5f94120b7304506042fe5bb/) (не подглядывайте!).

In [72]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
#plt.style.use("seaborn")

#### Описание данных

Автор д/з - плохой человек, который не стал переводить описание с мотивировкой, что весь DS на английском. Так что описание полей будет на английском:

1. Account ID
- Description: A unique identifier for each social media account in the dataset.
- Type: Integer
- Example: 1, 2, 3, …
2. Username
- Description: The username or handle of the social media account.
- Type: String
- Example: john_doe, tech_guru_22, fitness_freak
3. Platform
- Description: The social media platform the account is using (Instagram, Twitter, Facebook, TikTok, LinkedIn).
- Type: Categorical (String)
- Example: Instagram, Twitter, Facebook, TikTok, LinkedIn
4. Follower Count
- Description: The total number of followers the account has.
- Type: Integer
- Example: 1500, 245000, 78000
5. Posts Per Week
- Description: The average number of posts the account creates per week.
- Type: Integer
- Example: 3, 5, 7
6. Engagement Rate
- Description: The percentage of interactions (likes, comments, shares) relative to the follower count. This is a measure of how engaging the content is.
- Type: Float
- Range: 0.01 to 0.15
- Example: 0.045 (4.5% engagement rate)
7. Ad Spend (USD)
- Description: The monthly amount spent on advertising or promoting posts.
- Type: Float
- Example: 150.75, 850.00, 300.50
8. Conversion Rate
- Description: The percentage of users who take a desired action (e.g., clicking a link, signing up, etc.) after interacting with an ad.
- Type: Float
- Range: 0.01 to 0.05 (1% to 5% conversion rate)
- Example: 0.025 (2.5% conversion rate)
9. Campaign Reach
- Description: The total number of unique users reached by the user’s campaigns in a given month.
- Type: Integer
- Example: 5000, 20000, 15000

#### Задание 0

Подгрузите данные. Да-да, за чтение таблицы баллов не будет))

**Hint**: [pd.read_csv](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)

In [73]:
url = "https://raw.githubusercontent.com/poleno-ctrl/ALGO-HSE_25-26/main/ML/data.csv"
df = pd.read_csv(url)

#### Задание 1

Колонка `Platform` содержит название различных платформ. Давайте представим, что в них есть некоторое отношение порядка. Закодируйте каждую платформу целым числом (от 0 до N) и положите этот "код" в новую колонку `Platform_Code`. Теперь вычислите корреляцию Спирмена между всеми парами колонок в датасете (результатом будет таблица корреляций). В качестве ответа выведите значение корреляции `Platform_Code` с `Engagement Rate`. Можете после вывода числа еще коротко написать, что оно означает (нет, это не оценивается).

**Hint**: [pd.factorize](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.factorize.html), [pd.DataFrame.select_dtypes](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.select_dtypes.html), [pd.DataFrame.corr](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html).

In [74]:
df['Platform_Code'] = pd.factorize(df['Platform'])[0]
float(df.corr(numeric_only=True, method='spearman')['Platform_Code']['Engagement Rate'])

0.03138169529349812

#### Задание 2

Теперь посмотрите на столбец `Follower Count`. В нем какие-то числа. Иногда бывает полезно провести дискретизацию такого признака. Разбейте все значения в столбце на 4 группы: "Low", "Medium", "High", "Very High". Каждая группа включает в себя новые 25% данных. То есть, Low включает в себя 25% самых маленьких значений признака и так далее. Положите значения "Low", "Medium", "High" или "Very High" для каждого сэмпла датасета в новую колонку `Follower_Bin`. Теперь посчитайте среднее значение `Engagement Rate` для каждой категории из `Follower_Bin`. В качестве ответа выведите значение для категории "High".

**Hint**: [pd.qcut](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.qcut.html), [pd.groupby](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html), [pd.DataFrame.mean](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mean.html)

In [75]:
df['Follower_Bin'] = pd.qcut(df['Follower Count'], 4, labels=["Low", "Medium", "High", "Very High"])
float(df.groupby('Follower_Bin')['Engagement Rate'].mean()['High'])

0.08655032

#### Задание 3

Иногда бывает полезно превратить широкую таблицу в длинную (например, для визуализаций сразу нескольких признаков на одной картинке). Да, звучит странно, но именно этим вы сейчас и займетесь. Сделайте новый датафрейм `melted_df`, в который вы поместите каждый сэмпл датасета 6 раз: по одному разу на значение из 'Follower Count', 'Posts Per Week', 'Ad Spend (USD)', 'Conversion Rate', 'Engagement Rate' и 'Campaign Reach'. То есть, вы берете сэмпл из датасета (строку) и превращаете ее в 6 отдельных строк. Каждая отдельная строка в столбце `Metric` имеет имя из предложенного списка 5 признаков, а в столбце `Value` - значение данного сэмпла по этому признаку. Значение `Platform` повторяется в этих 6 строках.

Иначе говоря,

```json
{
    "Account ID": 1,
    "Username": "harrislisa",
    "Platform": "TikTok",
    "Follower Count": 54217,
    "Posts Per Week": 3,
    "Engagement Rate": 0.0986,
    "Ad Spend (USD)": 538.1,
    "Conversion Rate": 0.049,
    "Campaign Reach": 1308,
    "Platform_Code": 0,
    "Follower_Bin": "Low"
}
```

превращается в

```json
{
    "Platform": "TikTok",
    "Metric": "Follower Count",
    "Value": 54217,
},
{
    "Platform": "TikTok",
    "Metric": "Posts Per Week",
    "Value": 3,
}, ...
```

Для каждого уникальной пары значений (`Platform`, `Metric`) посчитайте моду среди всех значений `Value` для этой пары, результат сделайте списком и оставьте только наибольшее. В качестве ответа выведите сумму полученных мод (сумму всех значений в столбце `Value` уже после вычисления мод). Иначе говоря, выведите сумму всех мод значений для всех уникальных пар (`Platform`, `Metric`).

**Hint**: [pd.melt](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.melt.html), [pd.DataFrame.mode](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mode.html), [pd.DataFrameGroupBy.agg](https://pandas.pydata.org/docs/dev/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html)

In [76]:
melted_df = pd.melt(
    df,
    id_vars=['Platform'],
    value_vars=['Follower Count', 'Posts Per Week', 'Ad Spend (USD)', 'Conversion Rate', 'Engagement Rate', 'Campaign Reach'],
    var_name="Metric",
    value_name="Value"
)
vals = melted_df.groupby(['Platform', 'Metric'])['Value'].apply(lambda x: x.mode().max())
float(vals.sum())

3100285.4716

#### Задание 4

А теперь хочется посмотреть на самые популярные аккаунты на разных платформах. Для каждой платформы отсортируйте датафрейм по убыванию количества подписчиков (`Follower Count`) - да, без циклов, сразу для всех платформ сделать сортировку, а затем оставьте только первые три записи для каждой платформы - это и будут три самых популярных аккаунта для каждой платформы. В качестве ответа выведите саму таблицу и минимальное значение `Follower Count` в ней.

**Hint**: к *groupby* можно применять функции - это эквивалентно применению функции к каждой "группе" внутри groupby-объекта. Читайте [про применение apply к датафрейму после groupby](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html#flexible-apply).

In [77]:
df.groupby('Platform').apply(lambda x: x.sort_values('Follower Count', ascending=False).head(3))

Account ID         Username   Platform  Follower Count  \
Platform                                                                 
Facebook  2403        2404           eric65   Facebook          999982   
          7350        7351     patricknoble   Facebook          997915   
          1689        1690      chavezjason   Facebook          997512   
Instagram 8685        8686  alexandersamuel  Instagram          999726   
          3965        3966         lrodgers  Instagram          999351   
          2189        2190           jbrown  Instagram          997844   
LinkedIn  3039        3040          toneill   LinkedIn          999055   
          6359        6360    andrewgregory   LinkedIn          998968   
          2159        2160     ashleycooper   LinkedIn          998925   
TikTok    5838        5839     edwardthomas     TikTok          999739   
          4234        4235    andradewesley     TikTok          999234   
          2575        2576     williamwyatt     TikTok          998623   
Twitter   4920        4921      teresaellis    Twitter          999919   
          9684        9685           sriley    Twitter          999442   
          7576        7577       peggymunoz    Twitter          998216   

                Posts Per Week  Engagement Rate  Ad Spend (USD)  \
Platform                                                          
Facebook  2403               6           0.0642          884.06   
          7350               3           0.0834          429.01   
          1689               7           0.0834          993.20   
Instagram 8685               3           0.0834          687.61   
          3965               1           0.0834          565.07   
          2189               5           0.0642          505.61   
LinkedIn  3039               4           0.0642          799.49   
          6359               7           0.1020          797.64   
          2159               6           0.0856          474.46   
TikTok    5838               7           0.0642          630.77   
          4234               5           0.0834          872.77   
          2575               6           0.0856          477.98   
Twitter   4920               6           0.0834          411.63   
          9684               3           0.0834          206.84   
          7576               6           0.0642          456.61   

                Conversion Rate  Campaign Reach  Platform_Code Follower_Bin  
Platform                                                                     
Facebook  2403           0.0281           17312              2    Very High  
          7350           0.0182           25985              2    Very High  
          1689           0.0397           45717              2    Very High  
Instagram 8685           0.0205           11050              3    Very High  
          3965           0.0335           12391              3    Very High  
          2189           0.0202           14717              3    Very High  
LinkedIn  3039           0.0174           21862              1    Very High  
          6359           0.0351           15552              1    Very High  
          2159           0.0156           45956              1    Very High  
TikTok    5838           0.0325           35523              0    Very High  
          4234           0.0481           17188              0    Very High  
          2575           0.0250           43299              0    Very High  
Twitter   4920           0.0460            3975              4    Very High  
          9684           0.0225           12783              4    Very High  
          7576           0.0456           22037              4    Very High

#### Задание 5

Хочется посчитать какую-то метрику. Мы хотим посмотреть, на отношение разности суммы подписчиков аккаунтов с высокой и низкой конверсией к суммарному охвату рекламы на каждой платформе. То есть, мы делим аккаунты на две группы: высокая и низка конверсия. Затем мы смотрим на то, на сколько сильно влияние аккаунтов с высокой конверсией по сравнению с аккаунтами с низкой конверсией.

Давайте определим *Conversion Influence* следущим образом:

$$Conversion Influence = \frac{Total Follower\ Count (High) - Total Follower\ Count (Low)}{Total Campaign Reach (High)+Total Campaign Reach (Low)}$$

Считать эту метрику мы будет для каждой `Platform`. В этой формуле High - это значения всех сэмплов датасета, в которых `Conversion Rate` больше медианы, а `Low` - не более медианы. `Total Feature` - это суммарное количество значений `Feature` либо по `High` сэмплам, либо по `Low`.

Чтобы постоянно не пересчитывать, где High. где Low, сделайте новую колонку в датасете `Conversion_Category`. Положите в нее для каждой строки либо High, либо Low.

Выведите платформу с самым большим `Conversion Influence`.

**Hint**: данное задание не про *groupby*, а скорее про [pd.pivot_table](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.pivot_table.html). Сделайте сводную таблицу, по которой уже можно посчитать суммы, а затем подставить их в формулы.

In [78]:
df['Conversion_Category'] = df['Conversion Rate'].apply(lambda x: 'High' if x > df['Conversion Rate'].median() else 'Low')

table = pd.pivot_table(
    df,
    values=['Follower Count', 'Campaign Reach'],
    index='Platform',
    columns='Conversion_Category',
    aggfunc="sum"
)
conversion_influence_table = (table[('Follower Count', 'High')] - table[('Follower Count', 'Low')]) / (table[('Campaign Reach', 'High')] + table[('Campaign Reach', 'Low')])
conversion_influence_table.idxmax()

'Twitter'

#### Задание 6

Мы знаем, что вам понравилось считать метрики по формуле. Давайте закрепим этот успех. Теперь для каждой платформы посчитаем, на сколько эффективна реклама в разрезе трех последовательных записей в датасете.

Для каждой платформы отсортируйте записи в порядке убывания `Posts Per Week`. Будто бы аккаунты, которые постят чаще, используют более "активные" стратегии по рекламе. Теперь посчитайте *скользущие суммы с окном 3* по `Campaign Reach` и `Ad Spend (USD)`. Скользящая сумма с окном N - это вы идете по массиву, берете все последовательные тройки записей и суммируете их. Для первых двух записей троек не найдется. Для них скользящее среднее - NaN, что нам не помешает.

Теперь для каждого окна посчитайте

$$Rolling Efficiency Ratio = \frac{Rolling Sum of Campaign Reach}{Rolling Sum of Ad Spend}$$

По сути, для каждого окна вы посчитаете сколько пользователе привлеклось за один доллар, потреченный на рекламу, в данном окне. Понятно, что значений будет столько, сколько окон. Нам интересно максимально значение такой эффективности для каждой платформы.

В качестве ответа выведите название платформы с наибольшей максимальной эффективность и наименьшей (два названия, не одно, не три, ровно два).

**Hint**: окна можно делать через [pd.DataFrame.rolling](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html).

In [79]:
table = df.groupby('Platform').apply(lambda x: x.sort_values('Posts Per Week', ascending=False)[['Campaign Reach', 'Ad Spend (USD)']].rolling(3).sum())
table['Rolling Efficiency Ratio'] = table['Campaign Reach'] / table['Ad Spend (USD)']
highest_RER = table.groupby('Platform')['Rolling Efficiency Ratio'].max()
print(f"Наибольшая эффективность {highest_RER.idxmax()}")
print(f"Наименьшая эффективность {highest_RER.idxmin()}")

Наибольшая эффективность Facebook
Наименьшая эффективность LinkedIn


#### Задание 7

Это еще не все прекрасные функции pandas, которые мы хотим вам показать. Теперь вы посчитаете, сколько аккаунтов на каждой платформе одновременно лучшие по `Engagement Rate` и `Conversion Rate`.

Сделайте два отдельных суб-сета. В одном оставьте для каждой платфмормы один топовый аккаунт по `Engagement Rate`, в другом - по `Conversion Rate`. Соедините эти два подмножества по столбцу `Platform` так, что в одно строке есть описание сразу двух аккаунтов-лидеров. Теперь посмотрите равны ли имена аккаунтов в одной строке. Выведите количество строк, в которых названия аккаунтов совпадают.

In [80]:
ss1 = df.sort_values('Engagement Rate', ascending=False).groupby('Platform').head(1).set_index('Platform')
ss2 = df.sort_values('Conversion Rate', ascending=False).groupby('Platform').head(1).set_index('Platform')

res = ss1.join(ss2, lsuffix='_top_ER', rsuffix='_top_CR')
res[res['Username_top_CR'] == res['Username_top_ER']].shape[0]

0

#### Задание 8

Давайте теперь что-то попроще сделаем. Например, посчитаем отношение суммарного количества подписчиков на аккаунтах с высокой конверсией к такой же сумме в аккаунтах с низкой конверсией (очевидно, для каждой платформы). По сути, мы просто хотим получить число, которое характеризует, на сколько сильно аккаунты с высокой конверсией "доминируют" над аккаунтами с низкой конверсией в плане количества подписчиков.

Высокой конверсией будем считать конверсию больше средней. Остальное - низкая. Посчитайте суммы подписчиков для каждой платформы, поделите одно на другое и выведите разницу между самым большим значением и самым маленьким, а также платформы, которые соотвутствуют этим значениям.

Используйте магическую команду `%%time`, чтобы замерить, сколько времени ушло на исполнение вашего pandas-скрипта.

In [81]:
%%time
table = df.groupby('Platform').apply(
    lambda x:
              x[x['Conversion Rate'] > x['Conversion Rate'].mean()]['Follower Count'].sum() /
              x[x['Conversion Rate'] <= x['Conversion Rate'].mean()]['Follower Count'].sum()
    )

print(f"Разница: {table.max() - table.min()}")
print(f"Наименьшее: {table.idxmin()}")
print(f"Наибольшее: {table.idxmax()}")

Разница: 0.14886262659546368
Наименьшее: Instagram
Наибольшее: Twitter
CPU times: user 12.2 ms, sys: 0 ns, total: 12.2 ms
Wall time: 12 ms


#### Задание 9

А теперь решите задание 8 чисто питоном. Никаких функций и методов pandas. Только питоновские циклы. Замерьте время выполнения кода. Наконец, сравните время в задании 8 и 9. Напишите ниже, кто же победил: чистый python и pandas?

**Hint**: Чтобы итерироваться по датафрейму, можно из него сделать генератор через [pd.DataFrame.iterrows](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.iterrows.html) или [pd.DataFrame.itertuples](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.itertuples.html#pandas.DataFrame.itertuples). К слову, это не все способы итерироваться по датафрейму.

In [84]:
%%time

sum_d, count_d, high_d, low_d = {}, {}, {}, {}

for r in df.itertuples():
  sum_d[r.Platform] = sum_d.get(r.Platform, 0) + r._8
  count_d[r.Platform] = count_d.get(r.Platform, 0) + 1
  high_d[r.Platform] = low_d[r.Platform] = 0

m = {p: sum_d[p] / count_d[p] for p in sum_d}

for r in df.itertuples():
  if r._8 > m[r.Platform]:
    high_d[r.Platform] += r._4
  else:
    low_d[r.Platform] += r._4

rt = {p: high_d[p] / low_d[p] for p in high_d}
max1, min1 = max(rt, key=rt.get), min(rt, key=rt.get)

print(f"Разница: {max(rt.values()) - min(rt.values())}")
print(f"Наименьшее: {min1}")
print(f"Наибольшее: {max1}")

Разница: 0.14886262659546368
Наименьшее: Instagram
Наибольшее: Twitter
CPU times: user 47.4 ms, sys: 1.83 ms, total: 49.3 ms
Wall time: 49.7 ms


**А победителем является**: <А ТУТ МОЙ ОТВЕТ, Я ЗАМЕТИЛ, ЧТО В ЗАДАНИИ НУЖНО ЕЩЕ ЧТО-ТО НАПИСАТЬ ПОСЛЕ КОДА, ИНАЧЕ НЕ ПОЛУЧУ ПОЛНЫЙ БАЛЛ ЗА ЗАДАНИЕ>

`Итого pandas в 4 раза быстрее: 12.2 ms против 49.3 ms`

#### Задание 10

Крайне серьезное задание. Отнеситесь к нему соответствующе. В ячейке ниже напишите ваш любимый анекдот или мем (только без баянов, окей?). Можно плохие. Помните, это задание на полный балл. Проверяющий работу ассистент должен улыбнуться.

Если вставляете картинку, то убедитесь, что вы ее не подгружаете локально. А то будет неудобно - потерять балл на этом задании, когда надо было выложить картинку на облако и прокинуть ссылку. И нет, нельзя сюда просто ссылку вставить. Либо ищите, как вставить картинку, либо смешной анекдот. Есть всего два стула - выбирайте...

In [91]:
import gdown
from IPython.display import HTML
from base64 import b64encode

file_id = '1eMu-W0zwNMKEYkll5EVoLDZIHM1X0O-c'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'video.mp4'
gdown.download(url, output, quiet=True)

mp4 = open(output, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<div style="display: flex; justify-content: center;">
    <video height="600" controls autoplay muted loop>
        <source src="{data_url}" type="video/mp4">
    </video>
</div>
""")